# Exemple amb dades sintètiques

Aquest notebook comprova, amb dades on **sabem** la resposta, que el paquet `consum_temperatura` la recupera.
És la regla de la tesi: cap mètode s'aplica a dades reals abans de passar aquesta prova.

[![Obre a Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PauBuchacaMartin/consum-temperatura-urbana/blob/main/notebooks/exemple_dades_sintetiques.ipynb)

In [ ]:
# Instal·la el paquet directament des de GitHub (només cal a Colab)
!pip install -q git+https://github.com/PauBuchacaMartin/consum-temperatura-urbana.git

In [ ]:
import consum_temperatura as ct
from consum_temperatura import synthetic

# 1. Temperatura sintètica (5 anys) i consum amb una susceptibilitat que creix cada estiu
chi_real = {2021: 2.0, 2022: 2.5, 2023: 3.0, 2024: 3.5, 2025: 3.5}   # % per grau-dia
temp = synthetic.summer_temperature()
cons = synthetic.consumption(temp, tc=22.0, chi_by_year=chi_real)

In [ ]:
# 2. Temperatura de confort: la que minimitza l'error (hauria de sortir 22 °C)
tc, perfil = ct.choose_tc(cons, temp)
print("Tc estimada:", tc)
perfil.plot(x="tc", y="mse", marker="o", legend=False, ylabel="Error quadràtic mitjà");

In [ ]:
# 3. Susceptibilitat de cada estiu i creixement 2021 -> 2024
taula, beta, cov = ct.response_by_year(cons, temp, tc)
taula["real"] = taula.index.map(chi_real)
print(taula.round(2))
d, se = ct.growth(beta, cov, 2021, 2024)
print(f"Creixement 2021->2024: {d:+.2f} ± {se:.2f} %/K (real: +1.50)")

**Fixa't en una cosa important.** Si la $T_c$ estimada surt mig grau per sobre de la real, els **nivells** de $\chi^0$ queden esbiaixats (aquí, per sobre del valor real), però el **creixement** entre estius es manté. Per això l'article es basa en creixements i en el factor $\lambda$, no en els nivells absoluts. Prova de canviar la llavor (`seed`) de les dades sintètiques i mira com varia.

## Amb dades reals

Canvia el pas 1 per:

```python
temp = ct.from_cache("/content/drive/MyDrive/dades_tesi", "temperatura.csv",
                     lambda: ct.era5_daily(41.39, 2.17, "2021-01-01", "2025-12-31"))
cons = ...   # sèrie diària amb columnes 'data' i 'kwh' (vegeu ct.daily_consumption)
```

L'anàlisi completa de l'article és a `resposta_calor_bcn_v11.ipynb`.